# APIs, Integration & Backend Engineering — Hands-On

**Software Engineering · Week 04**

Offline notebook: simulate FastAPI-style contracts, auth, background jobs, and SQL using Python stdlib only.

## 0. Contract models and validation

In [ ]:
from dataclasses import dataclass, asdict
import asyncio, sqlite3, json

@dataclass(frozen=True)
class CreateTicketRequest:
    title: str
    priority: int
    tenant_id: str

@dataclass(frozen=True)
class TicketResponse:
    id: int
    title: str
    status: str

def parse_request(payload):
    title = payload.get("title")
    priority = payload.get("priority")
    tenant_id = payload.get("tenant_id")
    if not isinstance(title, str) or not title.strip():
        raise ValueError("title must be non-empty")
    if not isinstance(priority, int) or not 1 <= priority <= 5:
        raise ValueError("priority must be 1..5")
    if not isinstance(tenant_id, str) or not tenant_id:
        raise ValueError("tenant_id required")
    return CreateTicketRequest(title.strip(), priority, tenant_id)

## 1. Authentication vs authorization

In [ ]:
def authenticate(token):
    if not token or "sub" not in token:
        raise PermissionError("401 unauthenticated")
    return token

def authorize(principal, tenant_id, scope):
    if tenant_id not in principal.get("tenants", []):
        raise PermissionError("403 tenant forbidden")
    if scope not in principal.get("scopes", []):
        raise PermissionError("403 scope forbidden")

principal = authenticate({"sub":"u1", "tenants":["acme"], "scopes":["tickets:write"]})
authorize(principal, "acme", "tickets:write")
print("authorized", principal["sub"])

## 2. SQLite-backed handler with parameterized SQL

In [ ]:
db = sqlite3.connect(":memory:")
db.execute("CREATE TABLE tickets (id INTEGER PRIMARY KEY, tenant_id TEXT, title TEXT, priority INTEGER, status TEXT)")

def create_ticket_handler(payload, token):
    req = parse_request(payload)
    principal = authenticate(token)
    authorize(principal, req.tenant_id, "tickets:write")
    cur = db.execute(
        "INSERT INTO tickets(tenant_id, title, priority, status) VALUES (?, ?, ?, ?)",
        (req.tenant_id, req.title, req.priority, "open"),
    )
    db.commit()
    return TicketResponse(cur.lastrowid, req.title, "open")

resp = create_ticket_handler({"title":"Upload failed", "priority":3, "tenant_id":"acme"}, principal)
print(asdict(resp))

## 3. Error envelope

In [ ]:
def error_response(status, code, message, request_id):
    return {"status": status, "error": {"code": code, "message": message, "request_id": request_id}}

try:
    create_ticket_handler({"title":"", "priority":9, "tenant_id":"acme"}, principal)
except ValueError as exc:
    print(json.dumps(error_response(422, "VALIDATION_FAILED", str(exc), "req-1"), sort_keys=True))

## 4. Querying with SQL

In [ ]:
rows = db.execute(
    "SELECT tenant_id, status, COUNT(*) FROM tickets WHERE tenant_id=? GROUP BY tenant_id, status",
    ("acme",),
).fetchall()
print(rows)

## 5. Background job queue for slow work

In [ ]:
async def worker(queue, db):
    while True:
        item = await queue.get()
        if item is None:
            queue.task_done(); break
        job_id, filename = item
        result = f"indexed:{filename}"
        db.execute("UPDATE jobs SET status=?, result=? WHERE id=?", ("done", result, job_id))
        db.commit()
        queue.task_done()

async def run_jobs():
    job_db = sqlite3.connect(":memory:")
    job_db.execute("CREATE TABLE jobs (id INTEGER PRIMARY KEY, tenant_id TEXT, status TEXT, result TEXT)")
    queue = asyncio.Queue()
    task = asyncio.create_task(worker(queue, job_db))
    cur = job_db.execute("INSERT INTO jobs(tenant_id, status, result) VALUES (?, ?, ?)", ("acme", "queued", None))
    await queue.put((cur.lastrowid, "tickets.csv"))
    await queue.put(None)
    await queue.join()
    await task
    return job_db.execute("SELECT id, status, result FROM jobs").fetchall()

def run_coro(coro):
    # IPython kernels already run an event loop; run asyncio.run in a fresh thread.
    import threading
    result = []
    def target():
        result.append(asyncio.run(coro))
    thread = threading.Thread(target=target)
    thread.start(); thread.join()
    return result[0]

print(run_coro(run_jobs()))

## 6. Idempotency key for retryable writes

In [ ]:
db.execute("CREATE TABLE idempotency (key TEXT PRIMARY KEY, response TEXT)")

def once(key, make_response):
    row = db.execute("SELECT response FROM idempotency WHERE key=?", (key,)).fetchone()
    if row:
        return json.loads(row[0]), "replayed"
    response = make_response()
    db.execute("INSERT INTO idempotency(key, response) VALUES (?, ?)", (key, json.dumps(response)))
    db.commit()
    return response, "created"

print(once("req-abc", lambda: {"job_id": 1}))
print(once("req-abc", lambda: {"job_id": 2}))

## 7. API versioning as response shaping

In [ ]:
ticket = {"id": 1, "title": "Upload failed", "status": "open", "priority": 3}

def shape_response(ticket, version="v1"):
    if version == "v1":
        return {"id": ticket["id"], "title": ticket["title"], "status": ticket["status"]}
    if version == "v2":
        return {**shape_response(ticket, "v1"), "priority": ticket["priority"]}
    raise ValueError("unsupported version")

print("v1", shape_response(ticket, "v1"))
print("v2", shape_response(ticket, "v2"))

## Exercises
1. Add pagination parameters to the ticket list query.
2. Add a `403` test for a valid token from the wrong tenant.
3. Add a failed job state and retry counter.
4. Create a `v3` response that renames a field; why is that breaking?

## Links
- Literature note: `02 Literature Notes/Software Engineering/APIs, Integration & Backend Engineering`
- Snippets: `04 Code Snippets/Software Engineering/SE Week 04 Dataclass API Contract Handler`, `.../SE Week 04 Async Job Queue and SQLite Demo`
- MOC: `06 Maps of Content/Software Engineering Concepts`